# Synthetic SANS Machine Learning — A Physics-Informed Proof of Concept[¶](#Synthetic-SANS-Machine-Learning-%E2%80%94-A-Physics-Informed-Proof-of-Concept)

**Estimating the radius of homogeneous spherical nanoparticles using Kernel Ridge Regression (KRR)**

This notebook demonstrates a reproducible inverse-scattering workflow using *synthetic normalized sphere form factors* - not absolute experimental SANS intensities. It compares baseline and optimized KRR, evaluates robustness to 5% relative Gaussian noise, and tests noise-aware training.

**Central question:** Can a machine-learning model trained on noisy scattering data outperform one trained exclusively on idealized, noise-free data?

> **Scope:** This proof of concept focuses on noninteracting, homogeneous spherical nanoparticles with radii between 3 and 25 nm, using 200 scattering-vector points over the range 0.004–0.18 Å⁻¹. The analysis is based entirely on synthetic scattering data. The effects of restricted q-ranges and other experimental limitations are beyond the scope of the present study but may be investigated in future extensions.

## 1. Libraries and reproducibility[¶](#1.-Libraries-and-reproducibility)

All random seeds are fixed for repeatable results. No feature standardization is applied: the KRR models receive the original normalized form-factor vectors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
print(f"NumPy {np.__version__} | pandas {pd.__version__}")

# Shared figure style: Latin Modern Roman 10, vivid colors, and hollow markers.
from matplotlib import font_manager

for _font_path in font_manager.findSystemFonts():
    try:
        if font_manager.FontProperties(fname=_font_path).get_name() == "LM Roman 10":
            font_manager.fontManager.addfont(_font_path)
    except (OSError, RuntimeError):
        continue

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["LM Roman 10"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.edgecolor": "#000000",
    "axes.labelcolor": "#000000",
    "text.color": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "mathtext.fontset": "cm",
    "legend.fontsize": 10,
    "legend.frameon": False,
    "grid.color": "#bfbfbf",
    "grid.linewidth": 0.5,
    "lines.markerfacecolor": "none",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "svg.fonttype": "none",
})

BLUE = "#0000ff"     # RGB (0, 0, 255)
RED = "#ff0000"      # RGB (255, 0, 0)
CURVE_COLORS = [BLUE, RED, "#008000", "#ff00ff", "#000000"]

def fitted_limits(values, logarithmic=False, padding=0.04):
    """Fit an independent axis to finite data with a small visual margin."""
    values = np.asarray(values, dtype=float).ravel()
    values = values[np.isfinite(values)]
    if logarithmic:
        values = np.log10(values[values > 0])
    low, high = values.min(), values.max()
    span = high - low
    if span == 0:
        span = max(abs(low), 1.0) * 0.02
    low, high = low - padding * span, high + padding * span
    return (10**low, 10**high) if logarithmic else (low, high)

def finish_figure(fig, filename=None):
    """Keep the canvas compact while preserving labels and tick marks."""
    fig.tight_layout(pad=0.5)
    if filename is not None:
        fig.savefig(filename, dpi=300, bbox_inches="tight", pad_inches=0.04)
    plt.show()

## 2. Forward scattering model — sphere form factor[¶](#2.-Forward-scattering-model-%E2%80%94-sphere-form-factor)

For a homogeneous sphere of radius $R$:

$$
P(q;R)=\left[\frac{3\left(\sin(qR)-qR\cos(qR)\right)}{(qR)^3}\right]^2.
$$

Here $q$ is in inverse angestroms and $R$ must be converted to angestroms $$
1\,\mathrm{nm} = 10\,\text{Å}
$$
The form factor is dimensionless and normalized such that $P(0;R)=1$.

**This is a normalized shape-dependent factor, not the absolute measured intensity.** The normalized form factor (P(q)) describes the scattering pattern determined by particle geometry, independently of the overall scattering amplitude. In contrast, the physical intensity (I(q)) additionally depends on particle number density, scattering contrast, particle volume, and background contributions. In this proof of concept, we use normalized form factors to isolate the relationship between scattering-curve shape and nanoparticle radius, without modeling the full experimental intensity.

In [ ]:
def sphere_form_factor(q, radius_nm):
    """Normalized homogeneous-sphere form factor for q in 1/Å and radius in nm."""
    x = np.asarray(q, dtype=float) * (10.0 * radius_nm)
    amplitude = np.ones_like(x)
    large = np.abs(x) > 1e-4
    z = x[large]
    amplitude[large] = 3.0 * (np.sin(z) - z * np.cos(z)) / z**3
    s = x[~large]
    amplitude[~large] = 1.0 - s**2 / 10.0 + s**4 / 280.0
    return amplitude**2

# Physics sanity check: for R = 5 nm, first minimum near 4.493/(50 Å).
q_demo = np.linspace(0.001, 0.3, 1000)
r_demo = 5.0
p_demo = sphere_form_factor(q_demo, r_demo)
print(f"Expected first minimum: q ≈ {4.493/(10*r_demo):.4f} Å^-1")
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(q_demo, p_demo, color=BLUE, lw=1.4, marker="o",
        markevery=50, markersize=4.0, markerfacecolor="none",
        markeredgecolor=BLUE, markeredgewidth=1.0)
ax.set(xlabel=r"q (Å$^{-1}$)", ylabel="Normalized form factor P(q)",
       title="Analytical sphere form factor (R = 5 nm)")
ax.set_xlim(fitted_limits(q_demo))
ax.set_yscale("log")
ax.set_ylim(fitted_limits(p_demo, logarithmic=True))
ax.grid(True)
finish_figure(fig, "sans_00_sphere_form_factor.svg")

#### Sanity Check: Estimating the Nanoparticle Radius[¶](#Sanity-Check:-Estimating-the-Nanoparticle-Radius)

For a homogeneous sphere, the normalized form factor is

$$
P(q)=\left[3\frac{\sin(qR)-qR\cos(qR)}{(qR)^3}\right]^2
$$

This is precisely where the first zero of the spherical form factor occurs:

$$
q\_{\min}R \approx 4.493
$$

Therefore, the nanoparticle radius can be calculated as

$$
R=\frac{4.493}{q\_{\min}}
$$

Using the approximate position of the first minimum:

$$
q\_{\min} = 0.09 \; \text{Å}^{-1}
$$

we obtain

$$
R = \frac{4.493}{0.09} = 49.92 \; \text{Å}
$$

So:

$$
\boxed{R\_{\mathrm{estimated}}\approx4.99\ \mathrm{nm}}
$$

The original radius used to generate the scattering curve was

$$
R\_{\mathrm{input}}=5.00\ \mathrm{nm}
$$

The relative error is

$$
\mathrm{Relative\ Error}
=\frac{|R\_{\mathrm{estimated}}-R\_{\mathrm{input}}|}
{R\_{\mathrm{input}}}\times100
$$

$$
\boxed{\mathrm{Relative\ Error}\approx0.15\%}
$$

## 3. Synthetic inverse-problem dataset[¶](#3.-Synthetic-inverse-problem-dataset)

I generated 800 random radii uniformly between 3 and 25 nm. For each radius, the input vector contains 200 values of $P(q;R)$; the supervised target is $R$ (nm).

$$
\underbrace{P(q;R)}\_{\text{input curve}}\xrightarrow{\mathrm{KRR}}\underbrace{\hat R}\_{\text{estimated radius}}.
$$

In [ ]:
n_samples = 800
q = np.linspace(0.004, 0.18, 200)  # Å^-1
rng = np.random.default_rng(RANDOM_STATE)
y = rng.uniform(3.0, 25.0, n_samples)
X = np.array([sphere_form_factor(q, radius) for radius in y])
assert X.shape == (800, 200) and y.shape == (800,)
print("Features:", X.shape, "| radii:", y.shape)
print(f"Radius range: {y.min():.3f}–{y.max():.3f} nm")

## 4. Exploratory plot — sphere size and scattering features[¶](#4.-Exploratory-plot-%E2%80%94-sphere-size-and-scattering-features)

Larger spheres shift their first form-factor minimum toward lower $q$, approximately following $q\_\mathrm{min}R\approx4.493$. The selected curves are for visualization only; all 800 curves remain in the dataset.

In [ ]:
target_radii = [3, 5, 10, 15, 25]
indices = [int(np.argmin(np.abs(y - radius))) for radius in target_radii]
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for i, color in zip(indices, CURVE_COLORS):
    ax.plot(q, X[i], color=color, lw=1.2, marker="o", markevery=14,
            markersize=4.0, markerfacecolor="none", markeredgecolor=color,
            markeredgewidth=1.0, label=f"R = {y[i]:.2f} nm")
ax.set(xlabel=r"q (Å$^{-1}$)", ylabel="Normalized P(q)",
       title="Noise-free synthetic sphere form factors")
ax.set_yscale("log")
ax.set_xlim(fitted_limits(q))
ax.set_ylim(fitted_limits(X[indices], logarithmic=True))
ax.grid(True, which="major")
ax.set_title("Noise-free synthetic sphere form factors", pad=48)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=9)
finish_figure(fig, "sans_01_noise_free_curves.svg")

## 5. train-test-split[¶](#5.-train-test-split)

The same **640 training particles** and **160 test particles** are used throughout. Only the training set is used to select KRR hyperparameters; the test set is used for comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
print("Training:", X_train.shape, y_train.shape)
print("Test:    ", X_test.shape, y_test.shape)

## 6. Initial KRR baseline (noise-free training and testing)[¶](#6.-Initial-KRR-baseline-(noise-free-training-and-testing))

We use the radial basis function (RBF) kernel:

$$K(\mathbf{x}\_i,\mathbf{x}\_j)=\exp\left(-\gamma\|\mathbf{x}\_i-\mathbf{x}\_j\|^2\right).$$

The baseline values $\alpha=0.01$ and $\gamma=0.01$ are manually chosen and **not optimized**.

In [ ]:
krr_baseline = KernelRidge(kernel="rbf", alpha=0.01, gamma=0.01)
krr_baseline.fit(X_train, y_train)
y_pred_baseline = krr_baseline.predict(X_test)

## Evaluation of Kernel Ridge Regression[¶](#Evaluation-of-Kernel-Ridge-Regression)

**1. Mean Absolute Error (MAE)**

$$
MAE = \frac{1}{N}\sum\_{i=1}^{N}|R\_i-\hat{R}\_i|
$$

**2. Root Mean Squared Error (RMSE)**

$$
RMSE = \sqrt{\frac{1}{N}\sum\_{i=1}^{N}(R\_i-\hat{R}\_i)^2}
$$

**3. R-squared**

$$
R^2 = 1-\frac{\sum\_i(R\_i-\hat{R}\_i)^2}
{\sum\_i(R\_i-\bar{R})^2}
$$

Lower MAE and RMSE indicate better predictions, while an R-squared value closer to 1 indicates a better fit.

In [ ]:
def evaluate(y_true, y_hat):
    return {
        "MAE (nm)": mean_absolute_error(y_true, y_hat),
        "RMSE (nm)": np.sqrt(mean_squared_error(y_true, y_hat)),
        "R²": r2_score(y_true, y_hat)
    }

print(pd.Series(evaluate(y_test, y_pred_baseline)).to_string())

In [ ]:
# residuals
y_pred = y_pred_baseline
residuals = y_pred - y_test

# Predicted vs. Actual Radii
plt.close("all")
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(y_test, y_pred, s=27, marker="o", facecolors="none",
           edgecolors=BLUE, linewidths=1.0, label="Test samples")
limits = fitted_limits(np.concatenate([y_test, y_pred]))
ax.plot(limits, limits, color=RED, ls="--", lw=1.4,
        label="Perfect prediction")
ax.set(xlabel="Actual radius (nm)", ylabel="Predicted radius (nm)",
       title="KRR: Predicted vs. Actual Radii", xlim=limits, ylim=limits)
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper left")
ax.grid(True)
finish_figure(fig, "KRR_predicted_vs_actual.svg")

#### Observations[¶](#Observations)

- **Small particles (3–10 nm):** Predictions closely follow the diagonal, indicating very small prediction errors.
- **Intermediate particles (10–20 nm):** Predictions generally remain close to the ideal line, with minor deviations.
- **Large particles (20–25 nm):** Some predictions fall below the diagonal, indicating increasing underestimation of particle radii near the upper training boundary.

## 7. Optimize KRR on noise-free training data[¶](#7.-Optimize-KRR-on-noise-free-training-data)

Five-fold cross-validation chooses `alpha` and `gamma` by minimizing mean absolute error. Predictions on `X_test` are generated only after selecting hyperparameters.

In [ ]:
param_grid = {
    "alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1.0],
    "gamma": [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
}
search_clean = GridSearchCV(
    KernelRidge(kernel="rbf"), param_grid,
    cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
search_clean.fit(X_train, y_train)
krr_optimized = search_clean.best_estimator_
y_pred_optimized = krr_optimized.predict(X_test)
print("Best hyperparameters:", search_clean.best_params_)
print(f"CV MAE: {-search_clean.best_score_:.6f} nm")
print(pd.DataFrame({
    "Baseline": evaluate(y_test, y_pred_baseline),
    "Optimized (clean)": evaluate(y_test, y_pred_optimized)
}).to_string(float_format=lambda v: f"{v:.6f}"))

#### Performance Comparison[¶](#Performance-Comparison)

| Metric | Baseline KRR | Optimized KRR |
| --- | --- | --- |
| MAE (nm) | 0.066180 | 0.000928 |
| RMSE (nm) | 0.114215 | 0.002440 |
| R² | 0.999664 | ≈ 1.000000 |

#### Key Observations[¶](#Key-Observations)

- **MAE:** Hyperparameter optimization reduced the mean absolute error by approximately 98.6%, from 0.066180 nm to 0.000928 nm.
- **RMSE:** The root mean squared error decreased by approximately 97.9%, indicating a substantial reduction in larger prediction errors.
- **R²:** The optimized model achieved an R² extremely close to 1, demonstrating excellent agreement between predicted and actual nanoparticle radii.
- **Cross-validation:** The cross-validation MAE (0.001132 nm) is close to the held-out test MAE (0.000928 nm), suggesting consistent predictive performance across the investigated synthetic dataset.
- **Hyperparameter sensitivity:** The substantial improvement demonstrates that the predictive accuracy of KRR depends strongly on the selected regularization strength and RBF kernel parameter.

## 8. Baseline versus optimized KRR — residuals[¶](#8.-Baseline-versus-optimized-KRR-%E2%80%94-residuals)

Residual $e\_i=\hat R\_i-R\_i$ is positive for overestimation and negative for underestimation. Both panels use the same vertical scale so the reduction in error is visually comparable.

In [ ]:
res_baseline = y_pred_baseline - y_test
res_optimized = y_pred_optimized - y_test
for res, title, suffix in [
        (res_baseline, "Baseline KRR", "baseline"),
        (res_optimized, "Optimized KRR", "optimized")]:
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    ax.scatter(y_test, res, s=24, marker="o", facecolors="none",
               edgecolors=BLUE, linewidths=1.0, label="Test samples")
    ax.axhline(0, color=RED, lw=1.1, ls="--", label="Zero residual")
    ax.set(xlabel="True radius (nm)",
           ylabel="Predicted − true radius (nm)", title=title)
    ax.set_xlim(fitted_limits(y_test))
    ax.set_ylim(fitted_limits(np.append(res, 0.0)))
    ax.grid(True)
    ax.legend(loc="best")
    finish_figure(fig, f"sans_02_clean_residuals_{suffix}.svg")

#### Observations[¶](#Observations)

**1. Baseline KRR (Left Panel)**

- Prediction residuals exhibit a clear, systematic dependence on nanoparticle radius.
- **Small particles (3–10 nm):** Residuals remain close to zero, indicating relatively accurate predictions.
- **Intermediate particles (10–15 nm):** Negative residuals indicate slight underestimation.
- **Particles around 17–21 nm:** Positive residuals indicate systematic overestimation, reaching approximately +0.14 nm.
- **Large particles (22–25 nm):** Residuals become increasingly negative, reaching approximately −0.48 nm near the upper radius boundary.

The systematic residual pattern indicates that the baseline KRR model does not approximate the inverse scattering relationship equally well across the entire radius range.

**2. Optimized KRR (Right Panel)**

- Residuals are concentrated very close to zero across almost the entire radius range.
- The systematic overestimation and underestimation observed in the baseline model are substantially reduced.
- Minor deviations remain near the upper radius boundary (approximately 25 nm).
- The optimized model achieves an MAE of 0.000928 nm, compared with 0.066180 nm for the baseline.

Hyperparameter optimization substantially reduces both systematic prediction errors and residual magnitudes.

## 9. Stress-test the clean-trained model with 5% Gaussian relative noise[¶](#9.-Stress-test-the-clean-trained-model-with-5%25-Gaussian-relative-noise)

Noisy inputs are generated by

$$P\_{\mathrm{noisy}}(q)=\max\{0,P(q)[1+\varepsilon(q)]\},\quad\varepsilon\sim\mathcal N(0,0.05^2).$$

This is a simplified *multiplicative Gaussian* model, **not** a simulation of neutron-counting (Poisson) statistics. We perturb the 160 test curves only; the physical radii remain unchanged.

In [ ]:
noise_level = 0.05
rng_test_noise = np.random.default_rng(123)
X_test_noisy = np.clip(
    X_test * (1.0 + rng_test_noise.normal(0, noise_level, X_test.shape)),
    0, None
)
y_pred_noisy = krr_optimized.predict(X_test_noisy)
print(pd.DataFrame({
    "Optimized: clean test": evaluate(y_test, y_pred_optimized),
    "Optimized: 5% noisy test": evaluate(y_test, y_pred_noisy)
}).to_string(float_format=lambda v: f"{v:.6f}"))

## 10. Compare representative noisy scattering curves[¶](#10.-Compare-representative-noisy-scattering-curves)

The same test particles are used in the noise-free and noisy versions. The physical radius is unchanged.

In [ ]:
idx_test = [int(np.argmin(np.abs(y_test-r))) for r in target_radii]
for number, idx in enumerate(idx_test, start=1):
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ax.plot(q, X_test[idx], color=BLUE, lw=1.3, marker="o", markevery=14,
            markersize=4.0, markerfacecolor="none", markeredgecolor=BLUE,
            markeredgewidth=1.0, label="Noise-free")
    ax.plot(q, X_test_noisy[idx], color=RED, lw=1.0, marker="s",
            markevery=(7, 14), markersize=3.8, markerfacecolor="none",
            markeredgecolor=RED, markeredgewidth=1.0, label="5% noise")
    ax.set(xlabel=r"q (Å$^{-1}$)", ylabel="Normalized form factor",
           title=f"Representative SANS curves (R = {y_test[idx]:.2f} nm)")
    ax.set_yscale("log")
    ax.set_xlim(fitted_limits(q))
    ax.set_ylim(fitted_limits(np.concatenate([X_test[idx], X_test_noisy[idx]]),
                             logarithmic=True))
    ax.grid(True, which="major")
    ax.legend(loc="upper right")
    finish_figure(fig, f"sans_03_noise_example_{number}.svg")

## 11. Noise-aware KRR — separate training data and hyperparameter tuning[¶](#11.-Noise-aware-KRR-%E2%80%94-separate-training-data-and-hyperparameter-tuning)

Add an independent noise realization to the **640 training curves**, keeping their target radii fixed. Tune a second KRR model on the *noisy training set only*. Never train on or tune against the 160 noisy test curves.

In [ ]:
rng_train_noise = np.random.default_rng(456)
X_train_noisy = np.clip(
    X_train * (1.0 + rng_train_noise.normal(0, noise_level, X_train.shape)),
    0, None
)
param_grid_noisy = {
    "alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1.0],
    "gamma": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
}
search_noisy = GridSearchCV(
    KernelRidge(kernel="rbf"), param_grid_noisy,
    cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
search_noisy.fit(X_train_noisy, y_train)
krr_noise_aware = search_noisy.best_estimator_
y_pred_noise_aware = krr_noise_aware.predict(X_test_noisy)
print("Noise-aware parameters:", search_noisy.best_params_)
print(f"Noise-aware CV MAE: {-search_noisy.best_score_:.6f} nm")

## 12. Direct comparison on **identical** 5% noisy test curves[¶](#12.-Direct-comparison-on-identical-5%25-noisy-test-curves)

This is the central controlled comparison: noise-free-trained KRR versus noise-aware-trained KRR, evaluated on the same 160 noisy test curves.

In [ ]:
results = pd.DataFrame.from_dict({
    "Baseline KRR | clean → clean": evaluate(y_test, y_pred_baseline),
    "Optimized KRR | clean → clean": evaluate(y_test, y_pred_optimized),
    "Optimized KRR | clean → noisy": evaluate(y_test, y_pred_noisy),
    "Noise-aware KRR | noisy → noisy": evaluate(y_test, y_pred_noise_aware),
}, orient="index")
print(results.to_string(float_format=lambda v: f"{v:.6f}"))
mae_original_noisy = results.loc["Optimized KRR | clean → noisy", "MAE (nm)"]
mae_aware = results.loc["Noise-aware KRR | noisy → noisy", "MAE (nm)"]
print(f"Relative MAE reduction on noisy test: {(1-mae_aware/mae_original_noisy)*100:.2f}%")

## 13. Visual residual comparison under identical noisy conditions[¶](#13.-Visual-residual-comparison-under-identical-noisy-conditions)

Residuals on the right should be closer to zero if noise-aware training has improved robustness. Apparent bias must be quantified rather than assumed from visual appearance alone.

In [ ]:
res_clean_trained_noisy_test = y_pred_noisy-y_test
res_noise_aware = y_pred_noise_aware-y_test
for res, title, suffix in [
        (res_clean_trained_noisy_test, "Trained on noise-free curves", "clean_trained"),
        (res_noise_aware, "Trained on 5% noisy curves", "noise_aware")]:
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    ax.scatter(y_test, res, s=24, marker="o", facecolors="none",
               edgecolors=BLUE, linewidths=1.0, label="Test samples")
    ax.axhline(0, color=RED, lw=1.1, ls="--", label="Zero residual")
    ax.set(xlabel="True radius (nm)",
           ylabel="Predicted − true radius (nm)", title=title)
    ax.set_xlim(fitted_limits(y_test))
    ax.set_ylim(fitted_limits(np.append(res, 0.0)))
    ax.grid(True)
    ax.legend(loc="best")
    finish_figure(fig, f"sans_04_noise_aware_residuals_{suffix}.svg")
summary = pd.DataFrame({
    "Clean-trained on noisy test": [np.mean(res_clean_trained_noisy_test), np.std(res_clean_trained_noisy_test), np.max(np.abs(res_clean_trained_noisy_test))],
    "Noise-aware on noisy test": [np.mean(res_noise_aware), np.std(res_noise_aware), np.max(np.abs(res_noise_aware))],
}, index=["Mean signed error (nm)", "Residual std (nm)", "Max absolute error (nm)"])
print(summary.to_string(float_format=lambda v: f"{v:.6f}"))

## 14. Findings and limitations[¶](#14.-Findings-and-limitations)

**Reported findings in the original run**:

| Configuration | MAE (nm) | RMSE (nm) | R² |
| --- | --- | --- | --- |
| Baseline, clean → clean | 0.066180 | 0.114215 | 0.999664 |
| Optimized, clean → clean | 0.000928 | 0.002440 | approximately 1 |
| Optimized, clean → 5% noisy | 0.288812 | 0.341359 | 0.996995 |
| Noise-aware, 5% noisy → 5% noisy | 0.048699 | 0.067753 | 0.999882 |

**Interpretation.** Tuning improves the clean-data interpolation baseline. Testing that same model on 5% noisy curves leads to substantially larger errors. Training on separate 5% noisy versions of the original training particles improves performance on the **same** noisy test set (reported MAE reduction: **83.14%**).

**What these results do not demonstrate:** generalization to real SANS measurements; arbitrary particle morphologies; reliable prediction intervals; or statistically established superiority across repeated noise realizations. The models use the same analytical sphere model for all data, only one training noise realization per particle, and a simplified multiplicative Gaussian noise model. Hyperparameter changes and noise exposure both contribute to the observed improvement, so their individual effects are not isolated.